Evaluating on valset since COCO doesn't have a testset

In [5]:
"""
Evaluation Script for additive model
Evaluates BLEU, METEOR, ROUGE, and CIDEr on validation set.
"""

import os
import torch
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider


# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
MODEL_PATH = "/home/jovyan/lab3/additive-upgraded-20260511-013903.pt"
VAL_FEATURES_PATH = "/home/jovyan/annabell/val_features.pt"
VAL_PREPROCESSED_PATH = "/home/jovyan/lab3/coco_val_preprocessed_new.pt"


# ─────────────────────────────────────────────
# MODEL (additive)
# ─────────────────────────────────────────────
class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, max_length):
        super(ImageCaptionModel, self).__init__()
        self.image_embedding = nn.Sequential(
            nn.Linear(25088, 512),
            nn.ReLU(),
            nn.Dropout(0.3) 
        )
        self.caption_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=512
        )
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=512,
            batch_first=True
        )
        self.lstm_dropout = nn.Dropout(0.5)
        self.output_layer = nn.Linear(512, vocab_size)

    def forward(self, image_features, captions):
        img_emb = self.image_embedding(image_features)
        cap_emb = self.caption_embedding(captions)
        lstm_out, _ = self.lstm(cap_emb)
        lstm_out = self.lstm_dropout(lstm_out)
        img_expanded = img_emb.unsqueeze(1).repeat(1, lstm_out.size(1), 1)
        combined = lstm_out + img_expanded
        output = self.output_layer(combined)
        return output


# ─────────────────────────────────────────────
# CAPTION GENERATION (FIXED)
# ─────────────────────────────────────────────
def generate_caption(model, image_feature, vocab, idx_to_word, max_length=30, device="cpu"):
    model.eval()
    with torch.no_grad():
        img = image_feature.unsqueeze(0).to(device)
        img_emb = model.image_embedding(img)

        # Use 2D tensor for batch_first=True
        input_token = torch.tensor([[vocab["<start>"]]], device=device)
        

        hidden = None 
        words = []

        for _ in range(max_length):
            cap_emb = model.caption_embedding(input_token)
            
            # Pass and update hidden state
            lstm_out, hidden = model.lstm(cap_emb, hidden)

            combined = lstm_out + img_emb.unsqueeze(1)
            logits = model.output_layer(combined)

            # Get next token
            next_token = logits.argmax(dim=-1)
            word = idx_to_word.get(next_token.item(), "<unk>")

            if word == "<end>":
                break
            if word not in ("<start>", "<pad>"):
                words.append(word)

            # Update input_token for next step
            input_token = next_token

    return " ".join(words)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # ── Load model ─────────────────────────────
    checkpoint = torch.load(MODEL_PATH, map_location=device)

    vocab = checkpoint["vocab"]
    idx_to_word = {int(k): v for k, v in checkpoint["idx_to_word"].items()}
    max_length = checkpoint["max_length"]

    model = ImageCaptionModel(len(vocab), max_length).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # ── Load data ──────────────────────────────
    val_features = torch.load(VAL_FEATURES_PATH)
    val_data = torch.load(VAL_PREPROCESSED_PATH)
    val_samples = val_data["samples"]

    feature_lookup = {os.path.basename(k): v for k, v in val_features.items()}

    # ── CLEAN REFERENCES ───────────────────────
    refs_by_image = defaultdict(list)

    for s in val_samples:
        key = os.path.basename(s["image_path"])
        cap = s["caption"]
        if isinstance(cap, dict):
            cap = cap.get("caption", "")
        refs_by_image[key].append(str(cap))

    eval_keys = [k for k in refs_by_image if k in feature_lookup]
    print("Images:", len(eval_keys))

    # ── Generate captions ──────────────────────
    hypotheses = {}
    references = {}

    for k in tqdm(eval_keys, desc="Generating Captions"):
        feat = feature_lookup[k].float()
        hypotheses[k] = generate_caption(
            model, feat, vocab, idx_to_word, max_length, device
        )
        references[k] = refs_by_image[k]

    # ── BLEU ───────────────────────────────────
    print("\nBLEU:")
    smoother = SmoothingFunction().method1
    bleu_refs = [[r.split() for r in references[k]] for k in eval_keys]
    bleu_hyps = [hypotheses[k].split() for k in eval_keys]

    print("BLEU-1:", corpus_bleu(bleu_refs, bleu_hyps, weights=(1,0,0,0), smoothing_function=smoother))
    print("BLEU-2:", corpus_bleu(bleu_refs, bleu_hyps, weights=(0.5,0.5,0,0), smoothing_function=smoother))
    print("BLEU-3:", corpus_bleu(bleu_refs, bleu_hyps, weights=(0.33,0.33,0.33,0), smoothing_function=smoother))
    print("BLEU-4:", corpus_bleu(bleu_refs, bleu_hyps, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother))

    # ── METEOR ─────────────────────────────────
    print("\nMETEOR:")
    meteor_scores = [
        meteor_score([r.split() for r in references[k]], hypotheses[k].split())
        for k in eval_keys
    ]
    print(sum(meteor_scores) / len(meteor_scores))

    # ── ROUGE ──────────────────────────────────
    print("\nROUGE:")
    r_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []

    for k in eval_keys:
        best1 = best2 = bestL = 0
        for ref in references[k]:
            scores = r_scorer.score(ref, hypotheses[k])
            best1 = max(best1, scores["rouge1"].fmeasure)
            best2 = max(best2, scores["rouge2"].fmeasure)
            bestL = max(bestL, scores["rougeL"].fmeasure)
        r1.append(best1)
        r2.append(best2)
        rL.append(bestL)

    print("ROUGE-1:", sum(r1) / len(r1))
    print("ROUGE-2:", sum(r2) / len(r2))
    print("ROUGE-L:", sum(rL) / len(rL))

    # ── CIDEr ──────────────────────────────────
    print("\nCIDEr:")
    gts = {k: references[k] for k in eval_keys}
    res = {k: [hypotheses[k]] for k in eval_keys}
    cider = Cider()
    score, _ = cider.compute_score(gts, res)
    print(score)


if __name__ == "__main__":
    main()

Device: cuda
Images: 5000


Generating Captions: 100%|██████████| 5000/5000 [00:18<00:00, 267.94it/s]



BLEU:
BLEU-1: 0.5566898081584029
BLEU-2: 0.36622378488558394
BLEU-3: 0.23554718168578054
BLEU-4: 0.1497420428613171

METEOR:
0.36487951280882014

ROUGE:
ROUGE-1: 0.49422691604565444
ROUGE-2: 0.23096227356561172
ROUGE-L: 0.45584813106684535

CIDEr:
0.4446301950351274


In [1]:
"""
Evaluation Script for attention
Loads a saved model and evaluates it on the full val set using BLEU, METEOR, ROUGE, and CIDEr.
"""

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from tqdm import tqdm

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

# ─────────────────────────────────────────────
# PATHS (UNCHANGED)
# ─────────────────────────────────────────────
MODEL_PATH          = "/home/jovyan/lab3/attention-hidden-inject-20260511-141425.pt"
VAL_FEATURES_PATH   = "/home/jovyan/annabell/val_features.pt"
VAL_PREPROCESSED_PATH = "/home/jovyan/lab3/coco_val_preprocessed_new.pt"


# ─────────────────────────────────────────────
# MODEL DEFINITION (ATTENTION ARCHITECTURE)
# ─────────────────────────────────────────────
class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, max_length):
        super(ImageCaptionModel, self).__init__()

        self.image_embedding = nn.Sequential(
            nn.Linear(25088, 512),
            nn.ReLU(),
            nn.Dropout(0.3) 
        )

        self.caption_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=512
        )
        
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=512,
            batch_first=True
        )
        self.lstm_dropout = nn.Dropout(0.5)

        self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
        self.output_layer = nn.Linear(512, vocab_size)

    def forward(self, image_features, captions):
        img_emb = self.image_embedding(image_features)             # (batch, 512)
        cap_emb = self.caption_embedding(captions)                 # (batch, seq_len, 512)

        h0 = img_emb.unsqueeze(0)                                  # (1, batch, 512)
        c0 = torch.zeros_like(h0) 
        
        lstm_out, _ = self.lstm(cap_emb, (h0, c0))                 # (batch, seq_len, 512)
        lstm_out = self.lstm_dropout(lstm_out)

        img_context = img_emb.unsqueeze(1)                         # (batch, 1, 512)
        attn_output, _ = self.attention(query=lstm_out, key=img_context, value=img_context)
        
        combined = lstm_out + attn_output 
        return self.output_layer(combined)


# ─────────────────────────────────────────────
# CAPTION GENERATION
# ─────────────────────────────────────────────
def generate_caption(model, image_feature, vocab, idx_to_word, max_length=30, device="cpu"):
    model.eval()
    with torch.no_grad():
        img = image_feature.unsqueeze(0).to(device)
        img_emb = model.image_embedding(img)
        img_context = img_emb.unsqueeze(1)

        h = img_emb.unsqueeze(0) 
        c = torch.zeros_like(h)
        hidden = (h, c)

        input_token = torch.tensor([[vocab["<start>"]]], device=device)
        words = []

        for _ in range(max_length):
            cap_emb = model.caption_embedding(input_token)
            lstm_out, hidden = model.lstm(cap_emb, hidden)
            
            attn_out, _ = model.attention(query=lstm_out, key=img_context, value=img_context)
            logits = model.output_layer(lstm_out + attn_out)

            next_token = logits.argmax(dim=-1)
            word = idx_to_word.get(next_token.item(), "<unk>")

            if word == "<end>": break
            if word not in ("<start>", "<pad>"): words.append(word)
            input_token = next_token

    return " ".join(words)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # ── Load model ─────────────────────────────
    print(f"\nLoading model from {MODEL_PATH}...")
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

    vocab = checkpoint["vocab"]
    idx_to_word = {int(k): v for k, v in checkpoint["idx_to_word"].items()}
    max_length = checkpoint["max_length"]

    model = ImageCaptionModel(len(vocab), max_length).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print(f"  Vocab size: {len(vocab)}, max_length: {max_length}")

    # ── Load data ──────────────────────────────
    print("\nLoading val features and captions...")
    val_features = torch.load(VAL_FEATURES_PATH, weights_only=False)
    val_data = torch.load(VAL_PREPROCESSED_PATH, weights_only=False)
    val_samples = val_data["samples"]

    feature_lookup = {os.path.basename(k): v for k, v in val_features.items()}

    refs_by_image = defaultdict(list)
    for s in val_samples:
        key = os.path.basename(s["image_path"])
        cap = s["caption"]
        if isinstance(cap, dict):
            cap = cap.get("caption", "")
        refs_by_image[key].append(str(cap))

    eval_keys = [k for k in refs_by_image if k in feature_lookup]
    print(f"  {len(eval_keys)} images to evaluate")

    # ── Generate captions ──────────────────────
    print("\nGenerating captions...")
    hypotheses = {}
    references = {}

    for k in tqdm(eval_keys):
        feat = feature_lookup[k].float()
        hypotheses[k] = generate_caption(model, feat, vocab, idx_to_word, max_length, device)
        references[k] = refs_by_image[k]

    # ── Show 10 examples ──────────────────────
    print("\n── Sample generated captions ──")
    for key in list(hypotheses.keys())[:10]:
        print(f"  Image:     {key}")
        print(f"  Generated: {hypotheses[key]}")
        print(f"  Reference: {references[key][0]}")
        print()

    # ── BLEU ───────────────────────────────────
    print("Computing BLEU...")
    smoother = SmoothingFunction().method1
    bleu_refs = [[r.split() for r in references[k]] for k in eval_keys]
    bleu_hyps = [hypotheses[k].split() for k in eval_keys]

    bleu1 = corpus_bleu(bleu_refs, bleu_hyps, weights=(1, 0, 0, 0), smoothing_function=smoother)
    bleu2 = corpus_bleu(bleu_refs, bleu_hyps, weights=(0.5, 0.5, 0, 0), smoothing_function=smoother)
    bleu3 = corpus_bleu(bleu_refs, bleu_hyps, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoother)
    bleu4 = corpus_bleu(bleu_refs, bleu_hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoother)
    print(f"  BLEU-1: {bleu1:.4f}")
    print(f"  BLEU-2: {bleu2:.4f}")
    print(f"  BLEU-3: {bleu3:.4f}")
    print(f"  BLEU-4: {bleu4:.4f}")

    # ── METEOR ─────────────────────────────────
    print("\nComputing METEOR...")
    meteor_scores = [meteor_score([r.split() for r in references[k]], hypotheses[k].split()) for k in eval_keys]
    avg_meteor = sum(meteor_scores) / len(meteor_scores)
    print(f"  METEOR: {avg_meteor:.4f}")

    # ── ROUGE ──────────────────────────────────
    print("\nComputing ROUGE...")
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []

    for k in eval_keys:
        best_r1 = best_r2 = best_rL = 0
        for ref in references[k]:
            scores = scorer.score(ref, hypotheses[k])
            best_r1 = max(best_r1, scores["rouge1"].fmeasure)
            best_r2 = max(best_r2, scores["rouge2"].fmeasure)
            best_rL = max(best_rL, scores["rougeL"].fmeasure)
        r1.append(best_r1)
        r2.append(best_r2)
        rL.append(best_rL)

    print(f"  ROUGE-1: {sum(r1)/len(r1):.4f}")
    print(f"  ROUGE-2: {sum(r2)/len(r2):.4f}")
    print(f"  ROUGE-L: {sum(rL)/len(rL):.4f}")

    # ── CIDEr (FIXED) ──────────────────────────
    print("\nComputing CIDEr...")
    gts = {k: references[k] for k in eval_keys}
    res = {k: [hypotheses[k]] for k in eval_keys}
    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(gts, res)
    print(f"  CIDEr: {cider_score:.4f}")

    # ── Summary ────────────────────────────────
    print("\n══ Evaluation Summary ══")
    print(f"  BLEU-1:  {bleu1:.4f}")
    print(f"  BLEU-2:  {bleu2:.4f}")
    print(f"  BLEU-3:  {bleu3:.4f}")
    print(f"  BLEU-4:  {bleu4:.4f}")
    print(f"  METEOR:  {avg_meteor:.4f}")
    print(f"  ROUGE-1: {sum(r1)/len(r1):.4f}")
    print(f"  ROUGE-2: {sum(r2)/len(r2):.4f}")
    print(f"  ROUGE-L: {sum(rL)/len(rL):.4f}")
    print(f"  CIDEr:   {cider_score:.4f}")

if __name__ == "__main__":
    main()

Device: cuda

Loading model from /home/jovyan/lab3/attention-hidden-inject-20260511-141425.pt...
  Vocab size: 10307, max_length: 30

Loading val features and captions...
  5000 images to evaluate

Generating captions...


100%|██████████| 5000/5000 [00:36<00:00, 138.68it/s]



── Sample generated captions ──
  Image:     000000000139.jpg
  Generated: a living room with a fireplace and a tv
  Reference: A woman stands in the dining area at the table.

  Image:     000000000285.jpg
  Generated: a bear is standing in the grass with its tongue out
  Reference: A big burly grizzly bear is show with grass in the background.

  Image:     000000000632.jpg
  Generated: a living room with a couch and a window
  Reference: Bedroom scene with a bookcase, blue comforter and window.

  Image:     000000000724.jpg
  Generated: a stop sign with a street sign on it
  Reference: A stop sign is mounted upside-down on it's post. 

  Image:     000000000776.jpg
  Generated: a teddy bear sitting on top of a wooden table
  Reference: Three teddy bears, each a different color, snuggling together.

  Image:     000000000785.jpg
  Generated: a man is skiing down a snowy hill
  Reference: A woman posing for the camera standing on skis.

  Image:     000000000802.jpg
  Generated: a k

In [2]:
!pip install nltk


  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
